# Spatial accuracy analysis

In this notebook we analyze the spatial accuracy of detected transcripts for various spatial transcriptomics platforms assessed in our benchmark study. We focus on transcripts for the Ptgds gene. These are expressed in a thin cell layer in the tissue boundary.


## Python imports

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import gc
import numpy as np
import spatialdata as sd
from diffusion import LevelInfo, Experiment
from diffusion_plots import (
    DiffusionPlotSettings,
    plot_shapely_geometries,
    plot_tissue_outline_on_HE_image,
    plot_bins_colored_by_feature,
    plot_distance_kde_combined,
    plot_distance_kde_ridgeline,
    calculate_distance_histogram_hwhm,
    plot_distance_histogram_hwhm_bars,
    calculate_distance_to_kde_peak_median,
    plot_distance_to_kde_peak_median_bars,
)
from diffusion_utils import (
    compute_diffusion,
    generate_diffusion_plots,
    print_geodataframe_geometry_info,
)

/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiments definition

Info about experiments for which to perform spatial accuracy analysis.

In [3]:
SOURCE_PATH = "/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/"  # do *NOT* use the .zarr files in here directly

WORK_PATH = "/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion"
SDATA_PATH = os.path.join(WORK_PATH, "data")
PLOTS_PATH = os.path.join(WORK_PATH, "plots")

In [ ]:
# Choose boundary sections far enough apart that their eligible analysis bins are disjoint.
# Distances/counts are pooled per experiment without overlap detection or deduplication.
# Exactly identical endpoints, e.g. ((1000, 2000), (1000, 2000)), select the
# full nearest exterior ring in its stored order. Hole rings are excluded.
# Different boundary point pairs may select different geometries; each pair's endpoints must
# be nearest to the same exterior contour.
experiments = [
    Experiment("BMKMANU S1000",
               "Mouse brain",
               "exp74",
               "BMK_S1000/exp74/harpy/sdata.zarr",
               "exp74_he",
               0.341,  # Pixel size: 0.341 µm
               "exp74_tissue",
               [LevelInfo(f"level_{level}", f"exp74_level_{level}_table", f"exp74_level_{level}_labels") for level in [1, 2, 3, 4, 5, 6, 7, 13]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[#((16300, 15400), (17900, 7800)),
                                      ((16300, 15400), (16300, 15400)) # start==end means full boundary
                                    ]
                                    ),   
    Experiment("BMKMANU S3000",
               "Mouse brain",
               "exp99",
               "BMK_S3000/Exp99/harpy/sdata.zarr",
               "exp99_he",
               0.340,  # Pixel size: 0.340 µm
               "exp99_tissue", 
               [LevelInfo(f"exp99_level_{level}", f"exp99_level_{level}_table", f"exp99_level_{level}_labels") for level in [1,2,3,4,5,6,7,9,18]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[#((16000, 4500), (5000, 16500))
                                      ((16000, 4500), (16000, 4500))  # start==end means full boundary
                                    ]
                                     ),  
    Experiment("Nova-ST",
               "Mouse brain",
               "Exp65-sampleSPC014",
               "Nova-ST/Exp65-sampleSPC014/harpy/sdata.zarr",
               "Exp65-sampleSPC014_image",
               1.000,  # Pixel size: 1.000 µm
               "Exp65-sampleSPC014_tissue", 
               [LevelInfo("table", "table", "Exp65-sampleSPC014_labels")],    # OK? Or use hexagonal bins?
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[#((6000, 6000),(5500, 1000))
                                     ((5850,6000),(4700,6150)),
                                     ((4980,6150),(5600,6000))
                                     ]  
                                     ), 
    Experiment("Visium HD (FFPE)",
               "Mouse brain",
               "SCA003",
               "Visium_HD/SCA003/harpy/sdata.zarr",
               "SCA003_full_image",
               0.442,  # Pixel size: 0.442 µm
               "SCA003_tissue",
               [LevelInfo(f"square_{bin_size:03}um", f"square_{bin_size:03}um", f"SCA003_square_{bin_size:03}um_labels") for bin_size in [2, 8, 16, 32]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[#((2100, 9000), (11000, 13500))]
                                    #  ((13000, 3500), (11000, 13500)), # largest chunk
                                    #  ((14500, 6500), (15000, 4000)), # top right
                                    #  ((15000, 9000), (14400, 7000)), # right
                                     ((15000, 8500), (11000, 13500))
                                     ]
                                     ), 
                # Significant advection visible
    Experiment("Visium HD (FF)",
               "Mouse brain",
               "SCA019",
               "Visium_HD/SCA019/harpy/sdata.zarr",
               "SCA019_full_image",
               0.442,  # Pixel size: 0.442 µm
               "SCA019_tissue", 
               [LevelInfo(f"square_{bin_size:03}um", f"square_{bin_size:03}um", f"SCA019_square_{bin_size:03}um_labels") for bin_size in [2, 8, 16, 32]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[#((3000, 13500), (7800, 15800))
                                    ((8500, 2000),(1900, 11900)),
                                    ((2800, 13000),(10000, 12500))
                                    ]
                                     ),
    Experiment("Visium HD 3'",
               "Mouse brain",
               "SCA024",
               "Visium_HD/SCA024/harpy/sdata.zarr",
               "SCA024_full_image",
               0.442,  # Pixel size: 0.442 µm
               "SCA024_tissue",
               [LevelInfo(f"square_{bin_size:03}um", f"square_{bin_size:03}um", f"SCA024_square_{bin_size:03}um_labels") for bin_size in [2, 8, 16, 32]],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[
                                     ((5500,500),(5500,500))] # start==end means full boundary
                                     ), 
    Experiment("Stereo-Seq",
               "Mouse brain",
               "spc014",
               "Stereo-Seq/spc014/harpy/sdata.zarr",
               "spc014_image",
               0.500,  # Pixel size
               "spc014_tissue", 
               [LevelInfo("spc014_raw_table", "spc014_raw_table", "spc014_raw_labels")],
               LevelInfo("hexagonal_grid_20um", "hexagonal_grid_20um_table", "hexagonal_grid_20um_labels"),
               boundary_point_pairs=[#((14500, 11000), (5050, 15000))  # large section, diffusion calculations take > 8.5h
                                     #((13500, 7000), (4500, 8000))  # shorter section, more tractable
                                     ((13500, 7000), (13500, 7000))] # start==end means full boundary, will probably take 24h to calculate (!!)
                                     ),
]

# NOTE:
# No spatial accuracy analysis was performed for these platforms:
# - Visium FFPE (mouse brain, SCA016): large distance between capture spots
# - Curio Seeker: no microscope images available + tissue border (with Ptgds cell layer) outside capture area

In [5]:
GENE_NAME = "Ptgds"  # We will use this gene for diffusion analysis across datasets, it is expressed on the brain tissue boundary

MAX_DIFFUSION_DISTANCE_UM = 200.0  # Shared by the boundary preview and diffusion calculations


## Subset SpatialData objects

To avoid overwriting the SpatialData objects, we extract the subsets that we need and write them as new SpatialData objects.
These new objects will then be analyzed of diffusion.

In [6]:
MAKE_SUBSETTED_SDATA = False

if MAKE_SUBSETTED_SDATA:
    os.makedirs(SDATA_PATH, exist_ok=True)

    for experiment in experiments:
        # Read original sdata
        orig_zarr_path = os.path.join(SOURCE_PATH, experiment.sdata_path)
        print(f'Reading {orig_zarr_path}')
        sdata = sd.read_zarr(orig_zarr_path)
        print(sdata)

        # Copy the sdata object (to avoid overwriting it during diffusion analysis).
        # Keep only elements that are relevant for diffusion analysis while we're at it.
        keep_elements = [experiment.he_image, 
                         experiment.tissue_mask] + \
                        [experiment.hexbins.labels_layer,
                         experiment.hexbins.table_name] + \
                        [level.labels_layer for level in experiment.levels] + \
                        [level.table_name for level in experiment.levels] 

        print (f'Subsetting sdata -- keep elements {keep_elements}')

        sdata_subset = sdata.subset(
            element_names=keep_elements,
            filter_tables=False,
        )
        print(sdata_subset)

        # For some datasets, the tissue is fragmented and the tissue mask consists of disjunct regions.
        # These regions are stored as separate Polygons (rows) in the shapes layer's GeoDataframe,
        # but they happen to have the same index in the Dataframe. This identical index confuses the function _validate_polygons()
        # in spatialdata_plot (v0.2.14) and leads to only one Polygon being plotted.
        # Avoid this plotting problem by making the index unique here.
        if experiment.tissue_mask is not None:
            sdata_subset.shapes[experiment.tissue_mask].reset_index(inplace=True)

        # Write subsetted sdata to disk
        work_zarr_path = os.path.join(SDATA_PATH, experiment.sdata_path)
        print(f'Writing {work_zarr_path}')
        sdata_subset.write(work_zarr_path, overwrite=True)


## Closer look at the tissue outlines

Inspect how many disjunct polygons the tissue outline consists of. Ideally there is only one, but some samples consist of more than one tissue piece, and sometimes the tissue outline contains a couple of small polygons that are an artifact of tissue annotation in QuPath.

In [7]:
PLOT_CONTOUR_INFO = False

if PLOT_CONTOUR_INFO: 
    for experiment in experiments:
        print(f">>> {experiment.technology} {experiment.dataset_id} <<<")

        zarr_path = os.path.join(SDATA_PATH, experiment.sdata_path)
        print(f"Reading {zarr_path}")
        sdata = sd.read_zarr(zarr_path)

        output_folder = os.path.join(PLOTS_PATH, experiment.technology, experiment.dataset_id, "overview")
        os.makedirs(output_folder, exist_ok=True)

        print_geodataframe_geometry_info(sdata, experiment.tissue_mask)
        # Save the full mask and tissue_mask_geo{i}.png with the same full x/y range.
        plot_shapely_geometries(sdata,
                                experiment.tissue_mask,
                                output_path=os.path.join(output_folder, "tissue_mask.png"),
                                title=f"{experiment.technology}\nTissue mask")

## Overview images

In [9]:
PLOT_OVERVIEW_INFO = False

if PLOT_OVERVIEW_INFO:
    for experiment in experiments:
        print(f">>> {experiment.technology} {experiment.dataset_id} <<<")
        
        he_image = experiment.he_image
        tissue_mask = experiment.tissue_mask

        zarr_path = os.path.join(SDATA_PATH, experiment.sdata_path)
        print(f"Reading {zarr_path}")
        sdata = sd.read_zarr(zarr_path)
        print(sdata)

        output_folder = os.path.join(PLOTS_PATH, experiment.technology, experiment.dataset_id, "overview")
        os.makedirs(output_folder, exist_ok=True)

        if he_image is not None and tissue_mask is not None:
            gc.collect()
            filename = 'tissue.png'
            print(experiment)
            print(experiment.pixel_size_um)
            plot_tissue_outline_on_HE_image(sdata,
                                            he_image,
                                            tissue_mask,
                                            experiment.technology,
                                            experiment.dataset_id,
                                            experiment.pixel_size_um,
                                            os.path.join(output_folder, filename),
                                            fill_alpha=0.0,
                                            outline_width=1)

        gc.collect()
        print(f"> Adata for {experiment.hexbins}:")
        adata = sdata.tables[experiment.hexbins.table_name]
        print(adata)

        print(f"> {GENE_NAME} bins for {experiment.hexbins}")
        plot_bins_colored_by_feature(sdata,
                                     he_image,
                                     experiment.technology,
                                     experiment.dataset_id,
                                     experiment.pixel_size_um,
                                     experiment.hexbins,
                                     GENE_NAME,
                                     os.path.join(output_folder, f'{experiment.hexbins.name}_{GENE_NAME}.png'))
        
        for level in experiment.levels:
            gc.collect()
            print(f"> Adata for {level}:")
            adata = sdata.tables[level.table_name]
            print(adata)

            print(f"> {GENE_NAME} bins for {level}")
            plot_bins_colored_by_feature(sdata,
                                         he_image,
                                         experiment.technology,
                                         experiment.dataset_id,
                                         experiment.pixel_size_um,
                                         level,
                                         GENE_NAME,
                                         os.path.join(output_folder, f'{level.name}_{GENE_NAME}.png'))

## Diffusion analysis per technology

In [10]:
COMPUTE_DIFFUSION = False
GENERATE_DIFFUSION_PLOTS = True

CROP_MARGIN_PX = 200.0  # Visual margin for plots, irrelevant for analysis
MAX_INDIVIDUAL_PLOT_DIFFUSION_DISTANCE_UM = 100.0  # for plotting, close-up
BIN_WIDTHS_UM = [1.0, 2.0, 3.0, 4.0, 5.0, 10.0, 20.0]

if COMPUTE_DIFFUSION or GENERATE_DIFFUSION_PLOTS:
    for experiment in experiments:
        region_specs = experiment.boundary_point_pairs
        if region_specs:
            finest_level = experiment.levels[0]  # first level is the finest
            
            plot_settings = DiffusionPlotSettings(
                histogram_bin_widths_um=BIN_WIDTHS_UM,
                max_plot_distance_um=MAX_INDIVIDUAL_PLOT_DIFFUSION_DISTANCE_UM,
                tissue_outline=True,               # -> analysis/tissue.png (all sections)
                boundary=False,                    # -> analysis/crop{i}/boundary.png
                bin_gene_counts=False,             # -> analysis/crop{i}/transcripts_closeup.png
                full_tissue_bin_gene_counts=True,  # -> analysis/transcripts.png (all sections)
                bins_and_closest_points=False,     # -> analysis/crop{i}/closest_points.png
                distances_linearized=False,        # -> analysis/crop{i}/distances_linearized
                histogram=False,                   # -> analysis/pooled/histogram_binwidth_{bw}um.png
                boxplot=False                      # -> analysis/pooled/boxplot_um.png
            )

            max_diffusion_distance_px = MAX_DIFFUSION_DISTANCE_UM / experiment.pixel_size_um

            if COMPUTE_DIFFUSION:
                compute_diffusion(SDATA_PATH,
                                  experiment,
                                  level=finest_level.table_name,
                                  gene_name=GENE_NAME,
                                  plots_folder=PLOTS_PATH,
                                  max_distance_px=max_diffusion_distance_px,
                                  crop_margin_px=CROP_MARGIN_PX,
                                  use_signed_distances=True)   # calculate *signed* distances (left/right of tissue boundary, inside/outside of tissue)

            if GENERATE_DIFFUSION_PLOTS:
                generate_diffusion_plots(SDATA_PATH,
                                         experiment,
                                         level=finest_level.table_name,
                                         plots_folder=PLOTS_PATH,
                                         plot_settings=plot_settings)
        

Generating diffusion plots for BMKMANU S1000 exp74 at level exp74_level_1_table


no parent found for <ome_zarr.reader.Label object at 0x7f7a20314380>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a20302090>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a20314290>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a202edd60>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a202ec7a0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a202d26c0>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a202d0c80>: None
no parent found for <ome_zarr.reader.Label object at 0x7f7a202c7230>: None


best scale: scale1
figsize: (49.715, 50.0)
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'hexagonal_grid_120um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'hexagonal_grid_120um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'hexagonal_grid_120um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
Clipping input data to the valid range for imshow with RGB data 

INFO     Converted 1 Polygon(s) with holes to MultiPolygon(s) for correct rendering.                               


...

Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/Visium HD 3'/SCA024/analysis/tissue.png
Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/Visium HD 3'/SCA024/analysis/transcripts.png
Generating diffusion plots for Stereo-Seq spc014 at level spc014_raw_table


no parent found for <ome_zarr.reader.Label object at 0x7f7935047b00>: None
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'hexagonal_grid_120um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'spc014_bin_100_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'spc014_bin_10_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/d

best scale: scale2
figsize: (29.4, 29.4)
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      


/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'hexagonal_grid_120um_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'spc014_bin_100_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/.venv/lib/python3.12/site-packages/spatialdata/_core/spatialdata.py:170: UserWarning: The table is annotating 'spc014_bin_10_labels', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)
/data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diff

Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/Stereo-Seq/spc014/analysis/tissue.png
Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/Stereo-Seq/spc014/analysis/transcripts.png


## Diffusion analysis summary

All configured boundary sections are pooled per experiment before normalization, KDE peak detection/alignment, and summary statistics. Sections must select disjoint bins; no overlap detection or deduplication is performed.

Per-section results and spatial inspection plots remain in `analysis/crop{i}/`. Enabled individual histograms and boxplots are written to `analysis/pooled/`; comparison plots in `combined/` contain one distribution per experiment. Recompute diffusion after adding, changing, or reordering boundary pairs.

Each pair selects the exterior contour nearest to both endpoints; an error is raised if they select different contours. Different pairs may select different geometries. Exactly identical start/end coordinates select the full nearest exterior ring, starting at the snapped point and preserving vertex order. Hole boundaries are ignored. For full rings, signed distances are positive inside the exterior enclosure (including holes), negative outside, and zero on the boundary; no bins are excluded at the starting point.


In [11]:
MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM = 50.0

In [12]:
combined_dir = os.path.join(PLOTS_PATH, "combined")
os.makedirs(combined_dir, exist_ok=True)

In [13]:
# List only the currently configured sections; loaders pool these by (technology, dataset_id).
crop_specs = []
for experiment in experiments:   
    region_specs = experiment.boundary_point_pairs
    for i in range(len(region_specs)):
        crop_spec = (
            experiment.technology,
            experiment.dataset_id,
            experiment.pixel_size_um,
            f"crop{i+1}",
        )
        crop_specs.append(crop_spec)
print(crop_specs)

[('BMKMANU S1000', 'exp74', 0.341, 'crop1'), ('BMKMANU S3000', 'exp99', 0.34, 'crop1'), ('Nova-ST', 'Exp65-sampleSPC014', 1.0, 'crop1'), ('Nova-ST', 'Exp65-sampleSPC014', 1.0, 'crop2'), ('Visium HD (FFPE)', 'SCA003', 0.442, 'crop1'), ('Visium HD (FF)', 'SCA019', 0.442, 'crop1'), ('Visium HD (FF)', 'SCA019', 0.442, 'crop2'), ("Visium HD 3'", 'SCA024', 0.442, 'crop1'), ('Stereo-Seq', 'spc014', 0.5, 'crop1')]


### Histograms and ridgeline plots

In [14]:
for bin_width_um in BIN_WIDTHS_UM:
    bin_width_str = f"binwidth_{bin_width_um:g}um".replace(".", "p")
    print(f"Bin width {bin_width_um} um")

    output_path=os.path.join(PLOTS_PATH, "combined", bin_width_str, "distance_kde_um.png")
    plot_distance_kde_combined(
        crop_specs,
        output_path,
        PLOTS_PATH,
        GENE_NAME,
        MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
        align_peaks=False,
        bin_width_um=bin_width_um,
    )

    output_path=os.path.join(PLOTS_PATH, "combined", bin_width_str, "distance_kde_um_peaksaligned.png")
    plot_distance_kde_combined(
        crop_specs,
        output_path,
        PLOTS_PATH,
        GENE_NAME,
        MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
        align_peaks=True,
        bin_width_um=bin_width_um,
    )

    output_path=os.path.join(PLOTS_PATH, "combined", bin_width_str, "distance_kde_ridgeline_um_peaksaligned.png")
    plot_distance_kde_ridgeline(
        crop_specs,
        output_path,
        PLOTS_PATH,
        GENE_NAME,
        MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
        align_peaks=True,
        overlap=0.97,
    )

Bin width 1.0 um
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Bin width 2.0 um
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Bin width 3.0 um
Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), V

### Cumulative distribution function (CDF) plot

In [15]:
from diffusion_plots import plot_distance_cdf


plot_distance_cdf(
    crop_specs=crop_specs,
    output_path=os.path.join(combined_dir, "cdf_distance.png"),
    plots_root=PLOTS_PATH,
    max_distance_um=MAX_COMBINED_PLOT_DIFFUSION_DISTANCE_UM,
    origin="peak",
)

Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/combined/cdf_distance.png


### Median distance comparison

These median distances correspond to the abscis of the intersection between the CDF curves and y=0.5 (50%).

In [16]:
kde_peak_median_rows = calculate_distance_to_kde_peak_median(
    crop_specs=crop_specs,
    plots_root=PLOTS_PATH,
)

output_path = os.path.join(PLOTS_PATH, "combined", "median_distance.png")
plot_distance_to_kde_peak_median_bars(
    kde_peak_median_rows,
    output_path=output_path,
    title="Median distance to KDE peak",
)

print("label\tleft_median_distance_um\tright_median_distance_um\tpeak_um")
for row in kde_peak_median_rows:
    print(
        f"{row.label}\t{row.left_median_distance_um:.3f}\t{row.right_median_distance_um:.3f}\t{row.peak_position_um:.3f}"
    )

Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/combined/median_distance.png
label	left_median_distance_um	right_median_distance_um	peak_um
BMKMANU S1000	31.498	25.268	12.750
BMKMANU S3000	13.807	7.673	6.000
Nova-ST	12.845	13.320	6.500
Visium HD (FFPE)	22.541	12.489	1.250
Visium HD (FF)	3.846	4.746	5.500
Visium HD 3'	6.629	7.687	7.000
Stereo-Seq	14.750	16.250	12.750


### Textual summary

In [17]:
from diffusion_plots import summarize_distance_distributions

quantiles = (0.10, 0.25, 0.50, 0.75, 0.90)

summary_rows = summarize_distance_distributions(
    crop_specs=crop_specs,
    plots_root=PLOTS_PATH,
    quantiles=quantiles,
)


In [18]:
cdf_levels = (0.25, 0.50, 0.75, 0.90)

header = ["experiment", "outside_fraction", "inside_fraction"]
header.extend([f"outside_q{int(q*100):02d}_um" for q in quantiles])
header.extend([f"inside_q{int(q*100):02d}_um" for q in quantiles])
header.extend([f"outside_d_at_cdf_{int(q*100):02d}_um" for q in cdf_levels])
header.extend([f"inside_d_at_cdf_{int(q*100):02d}_um" for q in cdf_levels])
print("\t".join(header))
for row in summary_rows:
    fields = [
        row.label,
        f"{row.side_mass_fraction['outside_fraction']:.4f}",
        f"{row.side_mass_fraction['inside_fraction']:.4f}",
    ]
    fields.extend([f"{row.quantile_signature_um[f'outside_q{int(q*100):02d}_um']:.3f}" for q in quantiles])
    fields.extend([f"{row.quantile_signature_um[f'inside_q{int(q*100):02d}_um']:.3f}" for q in quantiles])
    fields.extend([f"{row.cdf_distance_um[f'outside_d_at_cdf_{int(q*100):02d}_um']:.3f}" for q in cdf_levels])
    fields.extend([f"{row.cdf_distance_um[f'inside_d_at_cdf_{int(q*100):02d}_um']:.3f}" for q in cdf_levels])
    print("\t".join(fields))


experiment	outside_fraction	inside_fraction	outside_q10_um	outside_q25_um	outside_q50_um	outside_q75_um	outside_q90_um	inside_q10_um	inside_q25_um	inside_q50_um	inside_q75_um	inside_q90_um	outside_d_at_cdf_25_um	outside_d_at_cdf_50_um	outside_d_at_cdf_75_um	outside_d_at_cdf_90_um	inside_d_at_cdf_25_um	inside_d_at_cdf_50_um	inside_d_at_cdf_75_um	inside_d_at_cdf_90_um
BMKMANU S1000	0.3952	0.6048	5.109	14.799	41.061	93.648	149.185	5.776	12.236	23.810	63.336	126.459	14.799	41.061	93.648	149.185	12.236	23.810	63.336	126.459
BMKMANU S3000	0.3458	0.6542	2.797	8.256	24.615	70.549	133.077	2.848	5.552	10.216	17.964	34.816	8.256	24.615	70.549	133.077	5.552	10.216	17.964	34.816
Nova-ST	0.3227	0.6773	1.866	6.153	18.486	49.696	115.434	3.068	6.920	14.164	35.263	119.179	6.153	18.486	49.696	115.434	6.920	14.164	35.263	119.179
Visium HD (FFPE)	0.5002	0.4998	1.497	5.208	27.881	79.191	131.116	1.285	3.348	11.150	60.978	142.410	5.208	27.881	79.191	131.116	3.348	11.150	60.978	142.410
Visium HD (FF)	0.1657	0.

### Half-width at half-maximum of distance histograms

This metric is used in [stereo-seq paper] but it does not capture the amount of diffusion very well. For example, if a lot of transcripts have diffused (or advected) outside the tissue (as in our Visium HD FFPE sample), the distance histograms (or the KDE curves) are still very symmetrical at half maximum. However, they have a lot more density towards the outside of the tissue than inwards, but this is only apparent at levels/counts much less than half maximum. So the asymmetry does not show up in the HWHM values.

In [19]:
align_peaks = True  # align KDE peaks before HWHM calculation

hwhm_rows = calculate_distance_histogram_hwhm(
    crop_specs=crop_specs,
    plots_root=PLOTS_PATH,
    align_peaks=align_peaks,
)

output_path = os.path.join(PLOTS_PATH, "combined", "hwhm_distance.png")
plot_distance_histogram_hwhm_bars(
    hwhm_rows,
    output_path=output_path,
    title=f"Half-width at half-maximum (HWHM) of {'peak-aligned ' if align_peaks else ''}distance KDE curves",
)

print("label\tleft_hwhm_um\tright_hwhm_um\tpeak_um")
for row in hwhm_rows:
    print(f"{row.label}\t{row.left_hwhm_um:.3f}\t{row.right_hwhm_um:.3f}\t{row.peak_position_um:.3f}")

Using shared KDE bandwidth: 1.469 µm for BMKMANU S1000, BMKMANU S3000, Nova-ST, Visium HD (FFPE), Visium HD (FF), Visium HD 3', Stereo-Seq
Saved /data/groups/technologies/spatial.catalyst/Frank/2024-07-UTBenchmark-SpC/diffusion/plots/combined/hwhm_distance.png
label	left_hwhm_um	right_hwhm_um	peak_um
BMKMANU S1000	15.143	13.072	0.000
BMKMANU S3000	5.413	8.593	0.000
Nova-ST	9.055	10.460	0.000
Visium HD (FFPE)	5.223	5.021	0.000
Visium HD (FF)	5.351	6.794	0.000
Visium HD 3'	8.411	9.642	0.000
Stereo-Seq	9.227	10.422	0.000


In [20]:
print("label\tleft_hwhm_um\tright_hwhm_um\tpeak_um")
for row in hwhm_rows:
    print(f"{row.label}\t{row.left_hwhm_um:.3f}\t{row.right_hwhm_um:.3f}\t{row.peak_position_um:.3f}")

label	left_hwhm_um	right_hwhm_um	peak_um
BMKMANU S1000	15.143	13.072	0.000
BMKMANU S3000	5.413	8.593	0.000
Nova-ST	9.055	10.460	0.000
Visium HD (FFPE)	5.223	5.021	0.000
Visium HD (FF)	5.351	6.794	0.000
Visium HD 3'	8.411	9.642	0.000
Stereo-Seq	9.227	10.422	0.000
